# 🔍 Object Detection using TensorFlow
### Classification + Localization on MNIST digits

This notebook trains a multi-output CNN that simultaneously:
- **Classifies** the digit (0–9)
- **Localizes** the digit with a bounding box

---
**Sections:**
1. Installing & Importing Libraries
2. Visualization Utilities
3. Loading and Preprocessing the Dataset
4. Define the Network
5. Train
6. Plot Metrics
7. Evaluate with IoU

## 1. Installing & Importing Libraries

In [ ]:
!pip install tensorflow tensorflow-datasets matplotlib pillow numpy --quiet

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless-safe; swap to 'TkAgg' for interactive display
import matplotlib.pyplot as plt
import PIL.Image
import PIL.ImageFont
import PIL.ImageDraw
import tensorflow as tf
import tensorflow_datasets as tfds

print(f"TensorFlow version: {tf.__version__}")

## 2. Visualization Utilities

In [ ]:
im_width  = 75
im_height = 75
use_normalized_coordinates = True

MATPLOTLIB_FONT_DIR = os.path.join(os.path.dirname(plt.__file__), "mpl-data/fonts/ttf")


def draw_bounding_boxes_on_image_array(image, boxes, color=[], thickness=1, display_str_list=()):
    """Wraps draw_bounding_boxes_on_image to work on numpy arrays."""
    image_pil = PIL.Image.fromarray(image.astype(np.uint8))
    rgbimg = PIL.Image.new("RGBA", image_pil.size)
    rgbimg.paste(image_pil)
    draw_bounding_boxes_on_image(rgbimg, boxes, color=color, thickness=thickness,
                                 display_str_list=display_str_list)
    return np.array(rgbimg)


def draw_bounding_boxes_on_image(image, boxes, color=[], thickness=1, display_str_list=()):
    """Draws bounding boxes on a PIL image."""
    box_shape = boxes.shape
    if not box_shape:
        return
    if (boxes.shape[0] != len(color)) or (boxes.shape[1] != 4):
        raise ValueError('boxes must be of shape [N, 4]')
    for i in range(box_shape[0]):
        draw_bounding_box_on_image(image,
                                   boxes[i, 1], boxes[i, 0],
                                   boxes[i, 3], boxes[i, 2],
                                   color[i], thickness, display_str_list[i])


def draw_bounding_box_on_image(image, ymin, xmin, ymax, xmax,
                                color='red', thickness=1,
                                use_normalized_coordinates=True):
    """Draws a single bounding box on a PIL image."""
    draw = PIL.ImageDraw.Draw(image)
    if use_normalized_coordinates:
        (left, right, top, bottom) = (xmin * im_width,  xmax * im_width,
                                      ymin * im_height, ymax * im_height)
    else:
        (left, right, top, bottom) = (xmin, xmax, ymin, ymax)
    draw.line([(left, top), (left, bottom), (right, bottom),
               (right, top), (left, top)], width=thickness, fill=color)


def dataset_to_numpy_util(training_dataset, validation_dataset, N):
    """Pulls one batch of size N from each dataset and returns numpy arrays."""
    val_batched   = validation_dataset.unbatch().batch(N)
    train_batched = training_dataset.unbatch().batch(N)

    if tf.executing_eagerly():
        for validation_digits, (validation_labels, validation_bboxes) in val_batched:
            validation_digits  = validation_digits.numpy()
            validation_labels  = validation_labels.numpy()
            validation_bboxes  = validation_bboxes.numpy()
            break
        for training_digits, (training_labels, training_bboxes) in train_batched:
            training_digits = training_digits.numpy()
            training_labels = training_labels.numpy()
            training_bboxes = training_bboxes.numpy()
            break

    validation_labels = np.argmax(validation_labels, axis=1)
    training_labels   = np.argmax(training_labels,   axis=1)
    return (training_digits,   training_labels,   training_bboxes,
            validation_digits, validation_labels, validation_bboxes)


def create_digits_from_local_fonts(n):
    """Renders digit images (0-9 cycling) using local matplotlib TTF fonts."""
    font_labels = []
    img = PIL.Image.new('LA', (75 * n, 75), color=(0, 255))
    try:
        font1 = PIL.ImageFont.truetype(os.path.join(MATPLOTLIB_FONT_DIR, 'DejaVuSans.ttf'),  25)
        font2 = PIL.ImageFont.truetype(os.path.join(MATPLOTLIB_FONT_DIR, 'STIXGeneral.ttf'), 25)
    except IOError:
        font1 = PIL.ImageFont.load_default()
        font2 = PIL.ImageFont.load_default()

    d = PIL.ImageDraw.Draw(img)
    for i in range(n):
        font_labels.append(i % 10)
        d.text((7 + i * 75, 4 if i < 10 else -4),
               str(i % 10),
               fill=(255, 255),
               font=font1 if i < 10 else font2)

    font_digits = np.array(img.getdata(), np.float32)[:, 0] / 255.0
    font_digits = np.reshape(
        np.stack(np.split(np.reshape(font_digits, [75, 75 * n]), n, axis=1), axis=0),
        [n, 75 * 75]
    )
    return font_digits, font_labels


def display_digits_with_boxes(digits, predictions, labels, pred_bboxes, bboxes, iou, title,
                               save_path=None):
    """
    Randomly samples 10 digits and plots them with predicted (red) and
    ground-truth (green) bounding boxes. Wrong predictions are shown in red.
    IoU score is printed below each image when available.
    """
    n = 10
    indexes       = np.random.choice(len(predictions), size=n, replace=False)
    n_digits      = digits[indexes]
    n_predictions = predictions[indexes]
    n_labels      = labels[indexes]

    n_iou         = iou[indexes]         if len(iou) > 0         else []
    n_pred_bboxes = pred_bboxes[indexes] if len(pred_bboxes) > 0 else []
    n_bboxes      = bboxes[indexes]      if len(bboxes) > 0      else []

    # FIX 1: Removed incorrect indentation on this line
    n_digits = n_digits * 255.0
    n_digits = n_digits.reshape(n, 75, 75)

    fig = plt.figure(figsize=(20, 4))
    plt.title(title)
    plt.xticks([])
    plt.yticks([])

    for i in range(10):
        ax = fig.add_subplot(1, 10, i + 1)
        bboxes_to_plot = []

        if len(n_pred_bboxes) > 0:
            bboxes_to_plot.append(n_pred_bboxes[i])
        if len(n_bboxes) > 0:
            bboxes_to_plot.append(n_bboxes[i])

        if bboxes_to_plot:
            img_to_draw = draw_bounding_boxes_on_image_array(
                image=n_digits[i],
                boxes=np.asarray(bboxes_to_plot),
                color=['red', 'green'],
                display_str_list=["True", "True"]
            )
            plt.imshow(img_to_draw)
        else:
            plt.imshow(n_digits[i], cmap='gray')

        plt.xlabel(str(n_predictions[i]))

        if n_predictions[i] != n_labels[i]:
            ax.xaxis.label.set_color('red')

        if len(n_iou) > 0:
            iou_val = float(n_iou[i][0])
            color   = 'green' if iou_val >= 0.5 else 'red'
            ax.text(0.2, -0.3, "iou: %s" % round(iou_val, 2),
                    color=color, transform=ax.transAxes)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    plt.show()
    plt.close(fig)

print("✅ Visualization utilities loaded.")

## 3. Loading and Preprocessing the Dataset

In [ ]:
strategy = tf.distribute.get_strategy()
print("Number of replicas in sync:", strategy.num_replicas_in_sync)

BATCH_SIZE = 64


def read_image_tfds(image, label):
    """
    Pads a 28x28 MNIST digit onto a blank 75x75 canvas at a random (xmin, ymin)
    offset, normalises to [0,1], and returns the image with a one-hot label and
    the normalised bounding box [xmin, ymin, xmax, ymax].
    """
    xmin = tf.random.uniform((), 0, 48, dtype=tf.int32)
    ymin = tf.random.uniform((), 0, 48, dtype=tf.int32)

    image = tf.reshape(image, [28, 28, 1])
    image = tf.image.pad_to_bounding_box(image, ymin, xmin, 75, 75)
    image = tf.cast(image, tf.float32) / 255.0

    xmin = tf.cast(xmin, tf.float32)
    ymin = tf.cast(ymin, tf.float32)
    xmax = xmin + 28
    ymax = ymin + 28

    # FIX 2: Removed stray blank line with inconsistent indentation
    xmin = xmin / 75
    xmax = xmax / 75
    ymin = ymin / 75
    ymax = ymax / 75

    return image, (tf.one_hot(label, 10), [xmin, ymin, xmax, ymax])


def get_training_dataset():
    with strategy.scope():
        dataset = tfds.load("mnist", split="train", as_supervised=True, try_gcs=True)
        dataset = dataset.map(read_image_tfds, num_parallel_calls=16)
        dataset = dataset.shuffle(5000, reshuffle_each_iteration=True)
        dataset = dataset.repeat()
        dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
        dataset = dataset.prefetch(1)
    return dataset


def get_validation_dataset():
    with strategy.scope():
        dataset = tfds.load("mnist", split="test", as_supervised=True, try_gcs=True)
        dataset = dataset.map(read_image_tfds, num_parallel_calls=16)
        dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
        dataset = dataset.prefetch(1)
    return dataset


print("[1/5] Loading datasets...")
training_dataset   = get_training_dataset()
validation_dataset = get_validation_dataset()

(training_digits,   training_labels,   training_bboxes,
 validation_digits, validation_labels, validation_bboxes) = dataset_to_numpy_util(
    training_dataset, validation_dataset, N=500
)
print(f"  training   — digits: {training_digits.shape},  labels: {training_labels.shape}")
print(f"  validation — digits: {validation_digits.shape}, labels: {validation_labels.shape}")

In [ ]:
OUTPUT_DIR = 'output_plots'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[2/5] Visualising training samples...")
display_digits_with_boxes(
    training_digits, training_labels, training_labels,
    np.array([]), training_bboxes, np.array([]),
    "Training Digits & Labels",
    save_path=os.path.join(OUTPUT_DIR, 'train_samples.png')
)

display_digits_with_boxes(
    validation_digits, validation_labels, validation_labels,
    np.array([]), validation_bboxes, np.array([]),
    "Validation Digits & Labels",
    save_path=os.path.join(OUTPUT_DIR, 'val_samples.png')
)

## 4. Define the Network

The model uses a **shared CNN backbone** feeding into two heads:
- 🔢 **Classification head** — softmax over 10 digit classes
- 📦 **Bounding box head** — 4 regression values `[xmin, ymin, xmax, ymax]`

In [ ]:
def feature_extractor(inputs):
    """Shared CNN backbone: 3x (Conv2D -> AveragePooling2D)."""
    # FIX 3: Moved docstring inside function (was floating outside as a bare string)
    x = tf.keras.layers.Conv2D(16, activation='relu', kernel_size=3,
                                input_shape=(75, 75, 1))(inputs)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(32, activation='relu', kernel_size=3)(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(64, activation='relu', kernel_size=3)(x)
    x = tf.keras.layers.AveragePooling2D((2, 2))(x)
    return x


def dense_layers(inputs):
    """Shared dense head on top of the CNN features."""
    x = tf.keras.layers.Flatten()(inputs)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    return x


def classifier(inputs):
    """10-class digit classification head (softmax)."""
    classification_output = tf.keras.layers.Dense(
        10, activation='softmax', name='classification')(inputs)
    return classification_output


def bounding_box_regression(inputs):
    """4-value bounding box regression head [xmin, ymin, xmax, ymax]."""
    bounding_box_regression_output = tf.keras.layers.Dense(
        4, name='bounding_box')(inputs)
    return bounding_box_regression_output


def final_model(inputs):
    """Combines backbone + both heads into a single multi-output Keras model."""
    feature_cnn           = feature_extractor(inputs)
    dense_output          = dense_layers(feature_cnn)
    classification_output = classifier(dense_output)
    bounding_box_output   = bounding_box_regression(dense_output)
    model = tf.keras.Model(inputs=inputs,
                           outputs=[classification_output, bounding_box_output])
    return model


def define_and_compile_model(inputs):
    """Builds and compiles the model with Adam, cross-entropy + MSE losses."""
    model = final_model(inputs)
    model.compile(
        optimizer='adam',
        loss={
            'classification': 'categorical_crossentropy',
            'bounding_box':   'mse'
        },
        metrics={
            'classification': 'accuracy',
            'bounding_box':   'mse'
        }
    )
    return model


print("[3/5] Building model...")
inputs = tf.keras.Input(shape=(75, 75, 1))
model  = define_and_compile_model(inputs)
model.summary()

## 5. Train

In [ ]:
print("[4/5] Training (20 epochs)...")
EPOCHS          = 20
STEPS_PER_EPOCH = 60000 // BATCH_SIZE  # full MNIST training set = 937 steps

history = model.fit(
    training_dataset,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
)

## 6. Plot Metrics

In [ ]:
def plot_metrics(metric_name, title, history, save_path=None):
    """Plots train and validation curves for a given metric key."""
    train_vals = history.history.get(metric_name, [])
    val_vals   = history.history.get('val_' + metric_name, [])
    epochs     = range(1, len(train_vals) + 1)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(epochs, train_vals, label='Train')
    ax.plot(epochs, val_vals,   label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    plt.show()
    plt.close(fig)


print("[5/5] Plotting metrics...")
plot_metrics('bounding_box_mse',        'Bounding Box MSE',        history,
             save_path=os.path.join(OUTPUT_DIR, 'metric_bbox_mse.png'))
plot_metrics('classification_accuracy', 'Classification Accuracy', history,
             save_path=os.path.join(OUTPUT_DIR, 'metric_cls_accuracy.png'))

## 7. Evaluate with IoU

**Intersection over Union (IoU)** measures how well the predicted bounding box overlaps the ground truth.
- IoU ≥ 0.5 → ✅ Good detection
- IoU < 0.5 → ❌ Poor detection

In [ ]:
def intersection_over_union(pred_box, true_box):
    """
    Computes Intersection-over-Union between predicted and ground-truth boxes.
    Both inputs are arrays of shape (N, 4): [xmin, ymin, xmax, ymax].
    Returns an array of shape (N, 1).
    """
    xmin_pred, ymin_pred, xmax_pred, ymax_pred = np.split(pred_box, 4, axis=1)
    xmin_true, ymin_true, xmax_true, ymax_true = np.split(true_box, 4, axis=1)

    smoothing_factor = 1e-10

    xmin_overlap = np.maximum(xmin_pred, xmin_true)
    xmax_overlap = np.minimum(xmax_pred, xmax_true)
    ymin_overlap = np.maximum(ymin_pred, ymin_true)
    ymax_overlap = np.minimum(ymax_pred, ymax_true)

    pred_box_area = (xmax_pred - xmin_pred) * (ymax_pred - ymin_pred)
    true_box_area = (xmax_true - xmin_true) * (ymax_true - ymin_true)

    overlap_area = (np.maximum((xmax_overlap - xmin_overlap), 0) *
                    np.maximum((ymax_overlap - ymin_overlap), 0))

    union_area = (pred_box_area + true_box_area) - overlap_area

    iou = (overlap_area + smoothing_factor) / (union_area + smoothing_factor)
    return iou


# Run predictions
predictions       = model.predict(validation_digits, batch_size=64)
predicted_labels  = np.argmax(predictions[0], axis=1)
prediction_bboxes = predictions[1]

iou = intersection_over_union(prediction_bboxes, validation_bboxes)

display_digits_with_boxes(
    validation_digits, predicted_labels, validation_labels,
    prediction_bboxes, validation_bboxes, iou,
    "True and Predicted Labels with Bounding Boxes",
    save_path=os.path.join(OUTPUT_DIR, 'predictions.png')
)

mean_iou = float(np.mean(iou))
val_acc  = history.history['val_classification_accuracy'][-1]

print(f"\n{'='*55}")
print(f"  Final val classification accuracy : {val_acc:.4f}")
print(f"  Mean IoU on validation set         : {mean_iou:.4f}")
print(f"  Plots saved to                     : {OUTPUT_DIR}/")
print(f"{'='*55}\n")